# Agentic AI System — Warehouse Intelligence Assistant (WIA)

**Author:** Sébastien Bodrero
**Programme:** Woolf University / Udacity MSc in Artificial Intelligence
**Module:** Agentic AI Systems (Module 6)
**Date:** April 2026

---

This notebook implements a single-agent system called the **Warehouse Intelligence Assistant (WIA)** — an LLM-powered agent that helps warehouse operations managers make real-time fleet coordination decisions. The agent uses the Claude API for reasoning, calls simulated warehouse tools, maintains a decision log in memory, and applies explicit safeguards before issuing any routing recommendation.

**Notebook structure:**
1. [Task 1 — Agentic Task and System Scope](#task1)
2. [Task 2 — Agent Architecture](#task2)
3. [Task 3 — Implementation](#task3)
4. [Task 4 — Execution and Observed Behavior](#task4)
5. [Task 5 — Summary](#task5)
6. [Task 6 — Report Reference](#task6)
7. [Task 7 — Requirements](#task7)

---
## Setup — Imports and Configuration

In [1]:
import os
import json
import datetime
from collections import deque
from typing import Any

import anthropic

print(f"anthropic SDK version : {anthropic.__version__}")
print(f"Notebook executed at  : {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

anthropic SDK version : 0.90.0
Notebook executed at  : 2026-04-19 18:14:13


<a id='task1'></a>
---
## Task 1 — Agentic Task and System Scope

### What the agent does

The **Warehouse Intelligence Assistant (WIA)** is a conversational agent that takes natural-language requests from a warehouse operations manager and translates them into concrete, safe fleet-coordination actions. A typical request might be:

> *"Route AGV-3 to pick zone C — what's the current inventory there and is the path clear?"*

The agent gathers context via tools (inventory levels, vehicle status, safe pathfinding), reasons about the best course of action, and produces a recommendation or escalates to a human operator when appropriate.

### Why an agentic approach is appropriate

A static classifier or rule engine cannot handle the **open-ended, multi-step nature** of warehouse coordination queries. The agent must:
- Decide *which* tools to call and *in what order* based on the query
- Integrate information across multiple tool results before reasoning
- Adapt its strategy when a tool returns unexpected results (e.g., zone blocked)
- Apply safety constraints that depend on runtime state, not static rules

This requires an LLM-driven reasoning loop — the hallmark of agentic design.

### Scope and boundaries

| In scope | Out of scope |
|---|---|
| Routing recommendations for named AGVs | Physically issuing commands to real hardware |
| Inventory level checks per zone | Persistent inventory database updates |
| Safe-path computation (BFS on a grid) | Full warehouse digital twin / simulation |
| Human escalation when safety zones are threatened | Multi-agent coordination between WIA instances |
| Decision logging for audit trail | Long-term persistent memory across sessions |

### Design type

**Single-agent** — one WIA instance handles one manager conversation. The agent is not persistent across sessions; each notebook execution is a fresh session. This scope is appropriate for the academic context and mirrors real-world "shift assistant" deployments where state is reset at shift change.

In [2]:
# ── Configuration constants ────────────────────────────────────────────────

# Zones where human operators are present; routing through these requires
# explicit escalation before the agent may recommend a path.
HUMAN_SAFETY_ZONES = {"H1", "H2", "H3", "MAIN_AISLE"}

# Maximum number of tool-call iterations per user turn (prevents runaway loops)
MAX_TOOL_ITERATIONS = 6

# Number of past decisions kept in the rolling decision log
MEMORY_WINDOW = 10

# Claude model to use for the agent's LLM backbone
MODEL_ID = "claude-haiku-4-5-20251001"   # fast & cost-efficient for this demo

print("Configuration loaded.")
print(f"  Safety zones   : {HUMAN_SAFETY_ZONES}")
print(f"  Max iterations : {MAX_TOOL_ITERATIONS}")
print(f"  Memory window  : {MEMORY_WINDOW}")
print(f"  LLM model      : {MODEL_ID}")


Configuration loaded.
  Safety zones   : {'MAIN_AISLE', 'H1', 'H3', 'H2'}
  Max iterations : 6
  Memory window  : 10
  LLM model      : claude-haiku-4-5-20251001
